In [0]:

# STEP 1: Locate the raw Repository Metadata files

# Define the location of our existing GitHub Raw Volume
root = "/Volumes/github_health/raw/bronze_data/bronze/"

# List all files and folders stored at this location
for item in dbutils.fs.ls(root):

    # Display each item's full path
    # We are looking for where repository metadata is stored
    print(item.path)


In [0]:

# STEP 2: Explore the Repository Metadata folder

# Path where raw repository metadata files are stored
metadata_path = "/Volumes/github_health/raw/bronze_data/bronze/repo_metadata/"

# List the files and subfolders inside the metadata folder
items = dbutils.fs.ls(metadata_path)

# Print the full path and size of every item
for item in items:
    print("Path:", item.path)
    print("Size:", item.size, "bytes")
    print("---------------------------")


In [0]:

# STEP 3: Find raw JSON files for all three repositories

# Main folder containing the repository metadata
metadata_path = "/Volumes/github_health/raw/bronze_data/bronze/repo_metadata/"

# Get the three repository subfolders
repo_folders = dbutils.fs.ls(metadata_path)

# Explore each repository folder
for folder in repo_folders:

    # Print the repository folder name
    print("\nRepository:", folder.name)

    # List the files stored inside this folder
    files = dbutils.fs.ls(folder.path)

    # Display each file's name, size and path
    for file in files:
        print("File:", file.name)
        print("Size:", file.size, "bytes")
        print("Path:", file.path)
        print("---------------------------")


In [0]:

# STEP 4: Inspect the structure of raw Repository Metadata

# Import Python's built-in library for reading JSON
import json

# Path to the Apache Spark repository metadata file
file_path = (
    "/Volumes/github_health/raw/bronze_data/bronze/"
    "repo_metadata/apache__spark/"
    "repo_metadata_full_2026-10-08_"
    "92199a9d-c132-4525-92c9-5094e8e1f310.jsonl"
)

# Read the existing JSONL file without changing it
file_content = dbutils.fs.head(file_path, 12000)

# JSONL stores one JSON object per line
# Extract and convert the first non-empty line into a Python object
first_line = next(
    line for line in file_content.splitlines() if line.strip()
)
record = json.loads(first_line)

# Display the type of the loaded record
print("Record type:", type(record).__name__)

# Display field names and their Python data types
# We avoid printing actual values or potentially sensitive information
if isinstance(record, dict):
    for field, value in record.items():

        print(f"{field}: {type(value).__name__}")

        # If a field contains another JSON object,
        # also show the names of its nested fields
        if isinstance(value, dict):
            print("  Nested fields:", list(value.keys()))


In [0]:

# STEP 5: Verify important metadata fields for all repositories

import json

# Folder containing the three repositories
base_path = "/Volumes/github_health/raw/bronze_data/bronze/repo_metadata/"

# Fields we plan to use in our Bronze and Silver processing
important_fields = [
    "id",
    "full_name",
    "stargazers_count",
    "forks_count",
    "open_issues_count",
    "subscribers_count",
    "language",
    "created_at",
    "updated_at",
    "pushed_at",
    "_repo",
    "_ingested_at",
    "_batch_id"
]

# Visit each repository folder
for folder in dbutils.fs.ls(base_path):

    # Find JSONL files in the repository folder
    for file in dbutils.fs.ls(folder.path):

        if not file.name.endswith(".jsonl"):
            continue

        # Read the raw JSONL file without modifying it
        content = dbutils.fs.head(file.path, 15000)

        # Convert each non-empty JSON line into a Python dictionary
        records = [
            json.loads(line)
            for line in content.splitlines()
            if line.strip()
        ]

        print("\nRepository folder:", folder.name)
        print("Records found:", len(records))

        # Show only the fields needed for our analytics
        for record in records:
            for field in important_fields:
                print(f"{field}: {record.get(field)}")

            print("-------------------------")


In [0]:

# STEP 1: Explore the raw Contributors dataset

# Path to the existing Contributors data in our Raw Volume
contributors_path = (
    "/Volumes/github_health/raw/bronze_data/bronze/contributors/"
)

# Get the repository folders containing Contributors data
repo_folders = dbutils.fs.ls(contributors_path)

# Explore each repository folder
for folder in repo_folders:

    # Print the repository folder name
    print("\nRepository:", folder.name)

    # List the files inside this repository folder
    for file in dbutils.fs.ls(folder.path):

        # Display the file name, size, and location
        print("File:", file.name)
        print("Size:", file.size, "bytes")
        print("Path:", file.path)
        print("-------------------------")


In [0]:

# STEP 2: Inspect the raw Contributors JSON structure

# Import Python's built-in JSON library
import json

# Path to the Contributors folder
contributors_path = (
    "/Volumes/github_health/raw/bronze_data/bronze/contributors/"
)

# Visit each repository folder
for folder in dbutils.fs.ls(contributors_path):

    print("\n" + "=" * 50)
    print("Repository:", folder.name)

    # Find the JSONL file inside this repository folder
    for file in dbutils.fs.ls(folder.path):

        if not file.name.endswith(".jsonl"):
            continue

        # Count records without modifying the source file
        record_count = spark.read.text(file.path).count()
        print("Total records:", record_count)

        # Read the beginning of the file for inspection
        content = dbutils.fs.head(file.path, 12000)

        # JSONL contains one JSON object per line
        first_line = next(
            line for line in content.splitlines()
            if line.strip()
        )

        # Convert the first JSON record into a Python dictionary
        record = json.loads(first_line)

        # Display field names and their data types
        # We avoid printing personal details or full JSON records
        print("\nFields and data types:")

        for field, value in record.items():
            print(f"{field}: {type(value).__name__}")

            # Show nested field names if a field contains an object
            if isinstance(value, dict):
                print("  Nested fields:", list(value.keys()))


In [0]:

%run ./schemas


In [0]:

# Check that our Contributors Raw schema is available
print("Contributors schema loaded:",
      "contributors" in RAW_SCHEMAS)


In [0]:

# STEP 5: Validate raw Contributor records

from pyspark.sql import functions as F

# Read all three Contributors JSONL files
contributors_path = (
    "/Volumes/github_health/raw/bronze_data/bronze/contributors/*/*.jsonl"
)

# Apply our explicit schema (no schema inference)
contributors_df = (
    spark.read
    .schema(RAW_SCHEMAS["contributors"])
    .json(contributors_path)
)

# Check required identifiers and collection timestamps
print("Total records:", contributors_df.count())

print(
    "Missing contributor IDs:",
    contributors_df.filter(F.col("id").isNull()).count()
)

print(
    "Missing repository names:",
    contributors_df.filter(
        F.col("_repo").isNull()
        | (F.trim(F.col("_repo")) == "")
    ).count()
)

print(
    "Invalid snapshot timestamps:",
    contributors_df.filter(
        F.expr("try_cast(_ingested_at AS TIMESTAMP)").isNull()
    ).count()
)

# The same GitHub account may contribute to multiple repositories.
# Therefore our logical key combines repository + contributor ID.
duplicate_keys = (
    contributors_df
    .groupBy("_repo", "id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate repository + contributor IDs:", duplicate_keys)
